# P19 ClaimIQ - Week 5: Bronze -> Silver Candidate Transformation

### ZENAIZ x BVRIT Hyderabad Data Engineering Internship

**Project:** P19 ClaimIQ
**Notebook:** `notebooks/03_silver_transformations.ipynb`
**Follows on from:** `notebooks/02_bronze_ingestion.ipynb` (Week 4)
**Method reference:** PageLoop Week 5 - Bronze -> Silver Candidate Transformation worked example
**Technology:** Databricks Free Edition - Spark SQL - Delta tables

### The Week 5 task

Week 4 created six ClaimIQ Bronze Delta tables: `bronze_claimiq_claims`,
`bronze_claimiq_policies`, `bronze_claimiq_policyholders`,
`bronze_claimiq_products`, `bronze_claimiq_providers` and
`bronze_claimiq_claim_payments`. Week 5 uses those six tables to build six
**Silver Candidate** tables that are typed, standardised and easier to
analyse, ready for the Week 6 data-quality assessment.

This notebook applies the **PageLoop Week 5 method** (standardise -> convert
types with `TRY_CAST` -> calculate documented columns -> write Candidate ->
validate) to the actual approved ClaimIQ project specification, exactly as
directed in Section 10 of the PageLoop worked example ("How to replicate this
for your assigned project").

One clear pipeline runs throughout:

`Bronze tables -> standardise existing columns -> convert data types -> calculate approved new columns -> write Silver Candidate -> validate`

Read the instruction above each code cell, run one cell at a time, inspect
the result, and complete the checkpoint before continuing.

## How to use this notebook

This is an executable Databricks lab, not a reading-only document.

1. Read the short explanation above a code cell.
2. Check that the catalog, schema and table names are correct.
3. Run only that code cell.
4. Inspect the Databricks result before moving forward.
5. Compare the result with the stated expected observation.
6. Complete the checkpoint or record the required evidence.

Do not use **Run all** on the first attempt. Run from top to bottom, one cell
at a time. After the notebook has completed successfully once, you may use
**Run all** for the controlled repeat-run test in Part 10.

### Week 4 handoff and Week 5 boundary

| Bronze (Week 4) | Silver Candidate (Week 5) |
|---|---|
| original source value remains unchanged | new typed or standardised representation |
| permanent ingestion evidence | derived Week 5 output |
| not judged for quality in Week 5 | not yet Trusted Silver |

This notebook reads the completed Bronze tables and writes new Candidate
tables. It does not update, delete, merge or filter Bronze rows, and it does
not join across entities - Week 5 keeps every Candidate table at its Bronze
entity's own physical grain.

## Week 5 objectives

By the end of this notebook you should be able to:

1. translate the approved ClaimIQ Silver specification into Spark SQL;
2. standardise identifiers and controlled categories without changing the underlying business fact;
3. use `TRY_CAST` to create correctly typed columns without losing physical rows;
4. create documented calculated columns from typed inputs for the claims entity;
5. build six persistent Silver Candidate Delta tables;
6. prove that record counts and Bronze lineage remain intact for all six entities;
7. prove a repeat run does not create unintended duplicate or missing records.

## Outcome first - what must ClaimIQ produce?

| Bronze input | Silver Candidate output | Main work |
|---|---|---|
| `bronze_claimiq_claims` | `silver_claimiq_claims_candidate` | standardise, type and calculate six new fields (worked example) |
| `bronze_claimiq_policies` | `silver_claimiq_policies_candidate` | standardise identifiers/status and type dates and amounts |
| `bronze_claimiq_policyholders` | `silver_claimiq_policyholders_candidate` | standardise identifiers/categories and type join date and active flag |
| `bronze_claimiq_products` | `silver_claimiq_products_candidate` | standardise identifiers/categories and type limits, SLA days and flags |
| `bronze_claimiq_providers` | `silver_claimiq_providers_candidate` | standardise identifiers/categories and type active flag and onboarding date |
| `bronze_claimiq_claim_payments` | `silver_claimiq_claim_payments_candidate` | standardise identifiers/status and type sequence, date and amount |

### Six new claims columns (worked example)

| New column | Simple meaning | Formula |
|---|---|---|
| `claim_duration_days` | total days the claim stayed open | closure timestamp - submission timestamp |
| `settlement_lag_days` | days taken to pay after a decision | max(settlement timestamp - decision timestamp, 0) |
| `is_closed` | whether status says closed | claim_status = `closed` |
| `is_paid` | whether status says paid | claim_status = `paid` |
| `calculated_net_payable` | amount actually payable to the claimant | approved amount - deductible amount |
| `reserve_variance` | difference between the reserved amount and the calculated net payable | reserve amount - calculated net payable |

These six fields are the **approved ClaimIQ Week 5 specification for the
claims entity**, adapted from the PageLoop loans worked example
(`actual_loan_days`, `days_overdue`, `is_returned`, `is_overdue`,
`calculated_net_fine`, `fine_variance`) to the claims lifecycle and money
facts.

### Completion proof

- all six Candidate tables exist as Delta tables;
- Bronze count = Candidate count for each of the six entities;
- every Candidate row retains the Bronze record hash, source file and ingestion run;
- safe-cast failures remain visible instead of deleting rows;
- counts remain unchanged after a controlled rerun.

# Part 1 - Confirm the Week 4 handoff

## 1.1 Select the working location

**Purpose:** use the same catalog and schema that hold the completed Week 4
ClaimIQ Bronze tables.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;

**Expected result:** one row showing `workspace` and `default`. If your
team used another approved location, change only these two settings and keep
every table reference in this notebook consistent with it.

## 1.2 Confirm the six ClaimIQ Bronze inputs

**Purpose:** verify that all six required ClaimIQ Bronze tables exist before
writing any Week 5 output.

In [ ]:
%sql
SHOW TABLES LIKE 'bronze_claimiq_*';

**Expected result:** `bronze_claimiq_claims`, `bronze_claimiq_policies`,
`bronze_claimiq_policyholders`, `bronze_claimiq_products`,
`bronze_claimiq_providers` and `bronze_claimiq_claim_payments` are all
listed. **Stop here** if any table is missing; repair Week 4 first.

## 1.3 Record the starting Bronze counts

**Purpose:** capture the Bronze row count for each entity. These counts
become the baseline for the Part 9 reconciliation.

In [ ]:
%sql
SELECT 'claims' AS entity, COUNT(*) AS bronze_rows FROM bronze_claimiq_claims
UNION ALL
SELECT 'policies', COUNT(*) FROM bronze_claimiq_policies
UNION ALL
SELECT 'policyholders', COUNT(*) FROM bronze_claimiq_policyholders
UNION ALL
SELECT 'products', COUNT(*) FROM bronze_claimiq_products
UNION ALL
SELECT 'providers', COUNT(*) FROM bronze_claimiq_providers
UNION ALL
SELECT 'claim_payments', COUNT(*) FROM bronze_claimiq_claim_payments;

**Expected result:** one count for each of the six entities. The actual
values depend on your Databricks run; this reference does not invent them.

# Part 2 - Understand the transformation patterns

Before building a complete table, learn the three SQL patterns used
throughout this notebook, shown here on the claims entity.

## Pattern A - standardise a controlled field

`TRIM` removes spaces at the beginning and end. `UPPER` or `LOWER` creates
one approved representation for identifiers and controlled codes.

In [ ]:
%sql
SELECT claim_id                    AS bronze_claim_id,
       upper(trim(claim_id))       AS candidate_claim_id,
       claim_status                AS bronze_claim_status,
       lower(trim(claim_status))   AS candidate_claim_status
FROM bronze_claimiq_claims
LIMIT 10;

**Expected result:** Bronze and Candidate representations appear side by
side. Do not apply casing changes to free-text fields (such as
`product_name`) unless the specification explicitly asks for it.

## Pattern B - convert a type safely

`TRY_CAST` converts valid values and returns `NULL` for a value it cannot
parse. The physical row remains present either way.

In [ ]:
%sql
SELECT submission_timestamp                             AS bronze_submission_timestamp,
       try_cast(submission_timestamp AS TIMESTAMP)      AS candidate_submission_timestamp,
       requested_amount                                 AS bronze_requested_amount,
       try_cast(requested_amount AS DECIMAL(12,2))      AS candidate_requested_amount
FROM bronze_claimiq_claims
LIMIT 10;

**Expected result:** successfully parsed values have proper types. An
invalid non-null Bronze value may become `NULL`; Week 5 records that outcome
but does not remove the row.

## Pattern C - create a documented calculated field

First type the input columns. Then calculate from those typed values. Never
invent a formula from a column name alone - use only the approved
specification.

In [ ]:
%sql
WITH example AS (
  SELECT try_cast(approved_amount AS DECIMAL(12,2))    AS approved_amount,
         try_cast(deductible_amount AS DECIMAL(12,2))  AS deductible_amount
  FROM bronze_claimiq_claims
)
SELECT approved_amount,
       deductible_amount,
       approved_amount - deductible_amount AS calculated_net_payable
FROM example
LIMIT 10;

**Expected result:** the inputs and calculated value are visible
together. If either input cannot be typed, the calculated value remains
`NULL` rather than being guessed.

# Part 3 - Build the claims Candidate table (worked example)

Claims is the anchor entity of ClaimIQ, so it is built in full detail here,
matching the depth of the PageLoop loans worked example. Build it in six
small stages:

1. standardise identifiers, categories and status;
2. convert dates, timestamps, amounts and the review flag;
3. inspect the typed result;
4. calculate durations and lifecycle flags;
5. calculate and compare the net payable amount;
6. inspect the calculations, check for parse failures and write the Delta table.

### 3.1 Standardise identifiers, categories and status

**Purpose:** create one consistent representation for every identifier and
controlled code, without changing any date, timestamp, amount or flag
value.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_standardised AS
SELECT
  upper(trim(source_record_id)) AS source_record_id,
  upper(trim(claim_id)) AS claim_id,
  upper(trim(policy_id)) AS policy_id,
  upper(trim(policyholder_id)) AS policyholder_id,
  upper(trim(product_id)) AS product_id,
  upper(trim(provider_id)) AS provider_id,
  lower(trim(claim_type)) AS claim_type,
  upper(trim(loss_category)) AS loss_category,
  loss_date, submission_timestamp, review_timestamp, decision_timestamp,
  settlement_timestamp, closure_timestamp,
  lower(trim(claim_status)) AS claim_status,
  upper(trim(outcome_code)) AS outcome_code,
  requested_amount, approved_amount, reserve_amount, deductible_amount,
  upper(trim(currency_code)) AS currency_code,
  upper(trim(risk_band)) AS risk_band,
  review_flag,
  upper(trim(exception_code)) AS exception_code,
  upper(trim(source_system)) AS source_system,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash, _rescued_payload
FROM bronze_claimiq_claims;

**Expected result:** `claims_standardised` is created. Identifiers and
controlled codes use one case each; every date, timestamp, amount and flag
value is still the untouched Bronze value, ready for typing in the next
stage.

### 3.2 Convert the documented data types

**Purpose:** convert dates, timestamps, amounts and the review flag safely.
`TRY_CAST` keeps the row even when a value cannot be parsed.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_typed AS
SELECT
  source_record_id, claim_id, policy_id, policyholder_id, product_id, provider_id,
  claim_type, loss_category,
  try_cast(loss_date AS DATE) AS loss_date,
  try_cast(submission_timestamp AS TIMESTAMP) AS submission_timestamp,
  try_cast(review_timestamp AS TIMESTAMP) AS review_timestamp,
  try_cast(decision_timestamp AS TIMESTAMP) AS decision_timestamp,
  try_cast(settlement_timestamp AS TIMESTAMP) AS settlement_timestamp,
  try_cast(closure_timestamp AS TIMESTAMP) AS closure_timestamp,
  claim_status, outcome_code,
  try_cast(requested_amount AS DECIMAL(12,2)) AS requested_amount,
  try_cast(approved_amount AS DECIMAL(12,2)) AS approved_amount,
  try_cast(reserve_amount AS DECIMAL(12,2)) AS reserve_amount,
  try_cast(deductible_amount AS DECIMAL(12,2)) AS deductible_amount,
  currency_code, risk_band,
  try_cast(review_flag AS BOOLEAN) AS review_flag,
  exception_code, source_system,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id,
  _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash,
  _rescued_payload
FROM claims_standardised;

**Expected result:** `claims_typed` is created. Business columns now
have the documented target types; lineage remains available under the
`_bronze_*` names.

### 3.3 Inspect the typed result

**Purpose:** verify the schema before using typed fields in calculations.

In [ ]:
%sql
DESCRIBE claims_typed;

**Expected result:** the five timestamp fields are `TIMESTAMP`,
`loss_date` is `DATE`, the four money fields are `DECIMAL(12,2)`,
`review_flag` is `BOOLEAN`, and the technical fields remain present.

### 3.4 Calculate durations and lifecycle flags

**Purpose:** add four documented fields using the typed timestamps and the
standardised claim status.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_with_time_measures AS
SELECT
  *,
  CASE WHEN submission_timestamp IS NOT NULL AND closure_timestamp IS NOT NULL
       THEN datediff(to_date(closure_timestamp), to_date(submission_timestamp)) END AS claim_duration_days,
  CASE WHEN decision_timestamp IS NOT NULL AND settlement_timestamp IS NOT NULL
       THEN greatest(datediff(to_date(settlement_timestamp), to_date(decision_timestamp)), 0) END AS settlement_lag_days,
  CASE WHEN claim_status IS NULL THEN NULL
       ELSE claim_status = 'closed' END AS is_closed,
  CASE WHEN claim_status IS NULL THEN NULL
       ELSE claim_status = 'paid' END AS is_paid
FROM claims_typed;

**Expected result:** the view has four new fields. If a required input
is null or unparseable, the related calculated value is null rather than
guessed.

### 3.5 Calculate and compare the net payable amount

**Purpose:** calculate the net payable amount from its components and
compare it with the reserve that was set aside for the claim.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW claims_candidate_ready AS
SELECT
  *,
  CASE WHEN approved_amount IS NOT NULL AND deductible_amount IS NOT NULL
       THEN cast(approved_amount - deductible_amount AS DECIMAL(12,2)) END AS calculated_net_payable,
  CASE WHEN reserve_amount IS NOT NULL AND approved_amount IS NOT NULL AND deductible_amount IS NOT NULL
       THEN cast(reserve_amount - (approved_amount - deductible_amount) AS DECIMAL(12,2)) END AS reserve_variance,
  current_timestamp() AS _candidate_created_at,
  'claimiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM claims_with_time_measures;

**Expected result:** the ready view contains all six new business
fields plus the Candidate creation time and schema version.
`reserve_variance = 0` means the reserved amount exactly matched the
calculated net payable; another value is retained for later investigation,
not corrected here.

### 3.6 Inspect the calculations

**Purpose:** view the typed inputs beside the six calculated outputs before
writing the table.

In [ ]:
%sql
SELECT claim_id, claim_status, submission_timestamp, decision_timestamp,
       settlement_timestamp, closure_timestamp,
       requested_amount, approved_amount, reserve_amount, deductible_amount,
       claim_duration_days, settlement_lag_days, is_closed, is_paid,
       calculated_net_payable, reserve_variance
FROM claims_candidate_ready
LIMIT 10;

**Expected result:** each calculated value can be explained directly
from the typed inputs shown in the same row.

Check only genuine parse failures: a non-null Bronze value that became null
after conversion.

In [ ]:
%sql
SELECT
  SUM(CASE WHEN submission_timestamp IS NOT NULL
           AND try_cast(submission_timestamp AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS submission_parse_failures,
  SUM(CASE WHEN requested_amount IS NOT NULL
           AND try_cast(requested_amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END) AS requested_amount_parse_failures,
  SUM(CASE WHEN review_flag IS NOT NULL
           AND try_cast(review_flag AS BOOLEAN) IS NULL THEN 1 ELSE 0 END) AS review_flag_parse_failures
FROM bronze_claimiq_claims;

**Expected result:** actual failure counts from your data. A non-zero
value is evidence for Week 6, not permission to delete or repair the row in
Week 5.

### 3.7 Write the claims Candidate table

**Purpose:** persist the completed claims result as a Delta table.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_claimiq_claims_candidate
USING DELTA
AS SELECT * FROM claims_candidate_ready;

**Expected result:** the claims Candidate Delta table is created.
`CREATE OR REPLACE` supports the controlled snapshot rerun used in Part 10.

In [ ]:
%sql
DESCRIBE TABLE silver_claimiq_claims_candidate;

In [ ]:
%sql
SELECT *
FROM silver_claimiq_claims_candidate
LIMIT 10;

# Part 4 - Build the policies Candidate table (guided practice)

Policies needs approved identifier/status standardisation plus type
conversion for its dates and amounts. It does not receive invented
calculated fields.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_claimiq_policies_candidate
USING DELTA
AS
SELECT
  upper(trim(source_record_id))                 AS source_record_id,
  upper(trim(policy_id))                        AS policy_id,
  upper(trim(policyholder_id))                  AS policyholder_id,
  upper(trim(product_id))                       AS product_id,
  trim(coverage_type)                           AS coverage_type,
  try_cast(policy_start_date AS DATE)           AS policy_start_date,
  try_cast(policy_end_date AS DATE)             AS policy_end_date,
  try_cast(coverage_limit AS DECIMAL(14,2))     AS coverage_limit,
  try_cast(deductible_amount AS DECIMAL(12,2))  AS deductible_amount,
  try_cast(premium_amount AS DECIMAL(12,2))     AS premium_amount,
  upper(trim(currency_code))                    AS currency_code,
  upper(trim(policy_status))                    AS policy_status,
  upper(trim(region_code))                      AS region_code,
  upper(trim(source_system))                    AS source_system,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'claimiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_claimiq_policies;

**Expected result:** a row-preserving policies Candidate table with
standardised identifiers/status, typed dates and amounts, and complete
Bronze lineage. `coverage_type` keeps its original casing because it is a
descriptive label, not a controlled code.

In [ ]:
%sql
SELECT policy_id, policyholder_id, product_id, coverage_type,
       policy_start_date, policy_end_date, coverage_limit,
       deductible_amount, premium_amount, policy_status, region_code,
       _source_file_name, _bronze_record_hash
FROM silver_claimiq_policies_candidate
LIMIT 10;

Check for genuine parse failures against the Bronze source.

In [ ]:
%sql
SELECT
  SUM(CASE WHEN policy_start_date IS NOT NULL
           AND try_cast(policy_start_date AS DATE) IS NULL THEN 1 ELSE 0 END) AS start_date_parse_failures,
  SUM(CASE WHEN coverage_limit IS NOT NULL
           AND try_cast(coverage_limit AS DECIMAL(14,2)) IS NULL THEN 1 ELSE 0 END) AS coverage_limit_parse_failures,
  SUM(CASE WHEN premium_amount IS NOT NULL
           AND try_cast(premium_amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END) AS premium_amount_parse_failures
FROM bronze_claimiq_policies;

**Checkpoint:** explain why each transformation is allowed. Only the
fields listed in the approved ClaimIQ Week 5 specification were
standardised or typed.

# Part 5 - Build the policyholders Candidate table (guided practice)

Policyholders needs approved identifier/segment standardisation plus type
conversion for `join_date` and `active_flag`.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_claimiq_policyholders_candidate
USING DELTA
AS
SELECT
  upper(trim(source_record_id))       AS source_record_id,
  upper(trim(policyholder_id))        AS policyholder_id,
  upper(trim(policyholder_segment))   AS policyholder_segment,
  upper(trim(age_band))               AS age_band,
  upper(trim(region_code))            AS region_code,
  upper(trim(risk_band))              AS risk_band,
  try_cast(join_date AS DATE)         AS join_date,
  try_cast(active_flag AS BOOLEAN)    AS active_flag,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'claimiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_claimiq_policyholders;

**Expected result:** a row-preserving policyholders Candidate table
with `join_date` as `DATE`, `active_flag` as `BOOLEAN`, and complete Bronze
lineage.

In [ ]:
%sql
SELECT policyholder_id, policyholder_segment, age_band, region_code,
       risk_band, join_date, active_flag,
       _source_file_name, _bronze_record_hash
FROM silver_claimiq_policyholders_candidate
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN join_date IS NOT NULL
           AND try_cast(join_date AS DATE) IS NULL THEN 1 ELSE 0 END) AS join_date_parse_failures,
  SUM(CASE WHEN active_flag IS NOT NULL
           AND try_cast(active_flag AS BOOLEAN) IS NULL THEN 1 ELSE 0 END) AS active_flag_parse_failures
FROM bronze_claimiq_policyholders;

**Expected result:** actual failure counts from your data. Keep any
affected row; Week 6 decides its data-quality outcome.

# Part 6 - Build the products Candidate table (guided practice)

Products needs approved identifier/category standardisation plus type
conversion for limits, the SLA target and the two flags. `product_name` is
free text and keeps its original casing.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_claimiq_products_candidate
USING DELTA
AS
SELECT
  upper(trim(source_record_id))                   AS source_record_id,
  upper(trim(product_id))                         AS product_id,
  trim(product_name)                              AS product_name,
  upper(trim(product_category))                   AS product_category,
  trim(coverage_type)                             AS coverage_type,
  try_cast(coverage_limit AS DECIMAL(14,2))       AS coverage_limit,
  try_cast(deductible_default AS DECIMAL(12,2))   AS deductible_default,
  try_cast(sla_target_days AS INT)                AS sla_target_days,
  try_cast(provider_required_flag AS BOOLEAN)     AS provider_required_flag,
  upper(trim(currency_code))                      AS currency_code,
  try_cast(active_flag AS BOOLEAN)                AS active_flag,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'claimiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_claimiq_products;

**Expected result:** a row-preserving products Candidate table with
`sla_target_days` as `INT`, both flags as `BOOLEAN`, both money columns as
`DECIMAL`, and complete Bronze lineage.

In [ ]:
%sql
SELECT product_id, product_name, product_category, coverage_type,
       coverage_limit, deductible_default, sla_target_days,
       provider_required_flag, active_flag,
       _source_file_name, _bronze_record_hash
FROM silver_claimiq_products_candidate
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN sla_target_days IS NOT NULL
           AND try_cast(sla_target_days AS INT) IS NULL THEN 1 ELSE 0 END) AS sla_target_days_parse_failures,
  SUM(CASE WHEN coverage_limit IS NOT NULL
           AND try_cast(coverage_limit AS DECIMAL(14,2)) IS NULL THEN 1 ELSE 0 END) AS coverage_limit_parse_failures
FROM bronze_claimiq_products;

**Expected result:** actual failure counts from your data. Products is
a small reference table, so review every row of the sample output, not just
a summary.

# Part 7 - Build the providers Candidate table (guided practice)

Providers needs approved identifier/type standardisation plus type
conversion for `active_flag` and `onboarding_date`.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_claimiq_providers_candidate
USING DELTA
AS
SELECT
  upper(trim(source_record_id))     AS source_record_id,
  upper(trim(provider_id))          AS provider_id,
  lower(trim(provider_type))        AS provider_type,
  upper(trim(provider_region))      AS provider_region,
  upper(trim(network_tier))         AS network_tier,
  try_cast(active_flag AS BOOLEAN)  AS active_flag,
  try_cast(onboarding_date AS DATE) AS onboarding_date,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'claimiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_claimiq_providers;

**Expected result:** a row-preserving providers Candidate table with
`active_flag` as `BOOLEAN`, `onboarding_date` as `DATE`, and complete Bronze
lineage.

In [ ]:
%sql
SELECT provider_id, provider_type, provider_region, network_tier,
       active_flag, onboarding_date,
       _source_file_name, _bronze_record_hash
FROM silver_claimiq_providers_candidate
LIMIT 10;

In [ ]:
%sql
SELECT COUNT(*) AS onboarding_date_parse_failures
FROM bronze_claimiq_providers
WHERE onboarding_date IS NOT NULL
  AND try_cast(onboarding_date AS DATE) IS NULL;

**Expected result:** an actual failure count. Keep any affected row;
Week 6 decides its data-quality outcome.

# Part 8 - Build the claim_payments Candidate table (guided practice)

Claim payments needs approved identifier/status standardisation plus type
conversion for the payment sequence, payment date and paid amount.
`provider_id` can be null on a payment row - `upper(trim(NULL))` correctly
stays `NULL` and the row is not dropped.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_claimiq_claim_payments_candidate
USING DELTA
AS
SELECT
  upper(trim(source_record_id))            AS source_record_id,
  upper(trim(payment_id))                  AS payment_id,
  upper(trim(claim_id))                    AS claim_id,
  try_cast(payment_sequence AS INT)        AS payment_sequence,
  try_cast(payment_date AS TIMESTAMP)      AS payment_date,
  upper(trim(payment_status))              AS payment_status,
  upper(trim(payment_method))              AS payment_method,
  try_cast(paid_amount AS DECIMAL(12,2))   AS paid_amount,
  upper(trim(currency_code))               AS currency_code,
  upper(trim(provider_id))                 AS provider_id,
  upper(trim(source_system))               AS source_system,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash, _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'claimiq_silver_candidate_v1.0' AS _candidate_schema_version
FROM bronze_claimiq_claim_payments;

**Expected result:** a row-preserving claim_payments Candidate table
with `payment_sequence` as `INT`, `payment_date` as `TIMESTAMP`,
`paid_amount` as `DECIMAL(12,2)`, and complete Bronze lineage.

In [ ]:
%sql
SELECT payment_id, claim_id, payment_sequence, payment_date,
       payment_status, payment_method, paid_amount, provider_id,
       _source_file_name, _bronze_record_hash
FROM silver_claimiq_claim_payments_candidate
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN payment_sequence IS NOT NULL
           AND try_cast(payment_sequence AS INT) IS NULL THEN 1 ELSE 0 END) AS payment_sequence_parse_failures,
  SUM(CASE WHEN payment_date IS NOT NULL
           AND try_cast(payment_date AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS payment_date_parse_failures,
  SUM(CASE WHEN paid_amount IS NOT NULL
           AND try_cast(paid_amount AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END) AS paid_amount_parse_failures
FROM bronze_claimiq_claim_payments;

**Expected result:** actual failure counts from your data. All six
Candidate tables now exist.

# Part 9 - Validate the complete Week 5 output

Validation answers two questions for every one of the six entities:

1. Did every Bronze physical row reach Candidate exactly once?
2. Can every Candidate row be traced back to Bronze?

## 9.1 Record reconciliation

In [ ]:
%sql
WITH counts AS (
  SELECT 'claims' AS entity,
         (SELECT COUNT(*) FROM bronze_claimiq_claims) AS bronze_rows,
         (SELECT COUNT(*) FROM silver_claimiq_claims_candidate) AS candidate_rows
  UNION ALL
  SELECT 'policies',
         (SELECT COUNT(*) FROM bronze_claimiq_policies),
         (SELECT COUNT(*) FROM silver_claimiq_policies_candidate)
  UNION ALL
  SELECT 'policyholders',
         (SELECT COUNT(*) FROM bronze_claimiq_policyholders),
         (SELECT COUNT(*) FROM silver_claimiq_policyholders_candidate)
  UNION ALL
  SELECT 'products',
         (SELECT COUNT(*) FROM bronze_claimiq_products),
         (SELECT COUNT(*) FROM silver_claimiq_products_candidate)
  UNION ALL
  SELECT 'providers',
         (SELECT COUNT(*) FROM bronze_claimiq_providers),
         (SELECT COUNT(*) FROM silver_claimiq_providers_candidate)
  UNION ALL
  SELECT 'claim_payments',
         (SELECT COUNT(*) FROM bronze_claimiq_claim_payments),
         (SELECT COUNT(*) FROM silver_claimiq_claim_payments_candidate)
)
SELECT *,
       candidate_rows - bronze_rows AS difference,
       CASE WHEN candidate_rows = bronze_rows THEN 'PASS' ELSE 'INVESTIGATE' END AS reconciliation_status
FROM counts
ORDER BY entity;

**Expected result:** `difference = 0` and `status = PASS` for all six
entities. A mismatch usually means a filter, `DISTINCT`, deduplication or an
unsafe join changed the grain.

## 9.2 Prove row-level lineage

A count match alone is not enough. Compare the Bronze record hashes with the
hashes retained in each Candidate table. Start with claims.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_claimiq_claims
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_claimiq_claims_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_claimiq_claims_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_claimiq_claims
  )) AS unexpected_candidate_rows;

**Expected result:** both claims values are zero.

Run the same row-level proof for policies.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_claimiq_policies
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_claimiq_policies_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_claimiq_policies_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_claimiq_policies
  )) AS unexpected_candidate_rows;

**Expected result:** both policies values are zero.

Run the same row-level proof for policyholders.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_claimiq_policyholders
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_claimiq_policyholders_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_claimiq_policyholders_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_claimiq_policyholders
  )) AS unexpected_candidate_rows;

**Expected result:** both policyholders values are zero.

Run the same row-level proof for products.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_claimiq_products
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_claimiq_products_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_claimiq_products_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_claimiq_products
  )) AS unexpected_candidate_rows;

**Expected result:** both products values are zero.

Run the same row-level proof for providers.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_claimiq_providers
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_claimiq_providers_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_claimiq_providers_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_claimiq_providers
  )) AS unexpected_candidate_rows;

**Expected result:** both providers values are zero.

Run the same row-level proof for claim_payments.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_claimiq_claim_payments
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_claimiq_claim_payments_candidate
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_claimiq_claim_payments_candidate
     EXCEPT ALL
     SELECT _record_hash FROM bronze_claimiq_claim_payments
  )) AS unexpected_candidate_rows;

**Expected result:** both claim_payments values are zero. All six
entities now have row-level lineage proof.

## 9.3 Confirm the output format

In [ ]:
%sql
SELECT table_name, data_source_format
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name LIKE 'silver_claimiq_%_candidate'
ORDER BY table_name;

**Expected result:** all six Candidate tables are listed with Delta as
the storage format. If this system view is unavailable in your edition, use
`DESCRIBE DETAIL <table_name>` on each table instead.

# Part 10 - Controlled repeat-run test

1. Record the current Candidate counts from Part 9.1.
2. Rerun the six `CREATE OR REPLACE TABLE` cells in Parts 3, 4, 5, 6, 7 and 8.
3. Run the Part 9.1 reconciliation cell again.

**Expected result:** counts remain stable. The table history may show a new
write, but the same snapshot must not be appended again.

> Do not claim repeat-run safety merely because the SQL completed. The proof
> is stable counts plus reconciliation, exactly as in the PageLoop worked
> example.

In [ ]:
%sql
DESCRIBE HISTORY silver_claimiq_claims_candidate;

**Checkpoint:** each rerun adds one `WRITE`/`CREATE OR REPLACE TABLE`
operation to the history, but the row count reported in Part 9.1 does not
grow. If it does, the write pattern silently appended instead of replacing -
stop and correct it before continuing to Week 6.

# Part 11 - Common failures and recovery

| Problem | Likely cause | Recovery |
|---|---|---|
| Bronze table not found | wrong catalog/schema, or Week 4 not run | select the correct location from Part 1.1; do not recreate Bronze here |
| Column not found | Bronze contract mismatch | run `DESCRIBE bronze_claimiq_<entity>`; resolve before changing Candidate SQL |
| Typed value becomes null | non-null Bronze value fails `TRY_CAST` | compare the value and target type; keep the row for Week 6 |
| Candidate count differs from Bronze | unintended filter, `DISTINCT`, deduplication or grain-changing join | remove it and rerun the affected `CREATE OR REPLACE TABLE` cell |
| Count grows after rerun | `INSERT`/`MERGE` append used instead of full replace | restore the `CREATE OR REPLACE TABLE ... AS SELECT` pattern from Parts 3-8 |
| Lineage is missing | Candidate view was not built from the `_bronze_*`-renamed metadata | inspect the standardised/typed view before writing the table |
| Tempted to fix an invalid fact (e.g. a negative amount) | trying to anticipate Week 6 | preserve it for Week 6 data-quality rules; do not guess in Week 5 |

After a correction, rerun from the affected entity's standardisation or
`CREATE OR REPLACE TABLE` cell and repeat its checks.

# Part 12 - Team ownership and evidence

| Student | Primary responsibility |
|---|---|
| Student A | verify the Week 4 Bronze handoff and document the six-entity Week 5 transformation contract |
| Student B | implement the six Candidate tables and the claims calculated fields |
| Student C | run reconciliation, row-level lineage and the controlled rerun; organise evidence |

All three students must review the final notebook and be able to explain at
least one standardisation, one type conversion, one calculated field and one
validation.

### Required repository evidence

- executed project notebook at `notebooks/03_silver_transformations.ipynb`;
- Candidate schema/sample evidence for all six entities;
- count reconciliation evidence (Part 9.1, all `PASS`);
- row-level lineage evidence (Part 9.2, all zero);
- controlled rerun evidence (Part 10, stable counts);
- Week Log with Student A/B/C contributions;
- AI Transparency Note stating what AI suggested and what the team verified.

Never submit fabricated results, credentials, workspace tokens or private
URLs.

# Part 13 - Week 5 exit checklist

The team is ready to close Week 5 only when every statement is true:

- [ ] All six ClaimIQ Bronze inputs were confirmed before any Candidate table was written.
- [ ] Only documented standardisations (identifier/controlled-code casing and trimming) were applied - no free-text field was recased.
- [ ] Every date, timestamp, amount and flag conversion used `TRY_CAST`, never a forcing `CAST`.
- [ ] All six claims calculated fields use typed inputs and the approved formulas.
- [ ] No rows were filtered, deduplicated, rejected or quarantined in any of the six Candidate tables.
- [ ] Bronze count equals Candidate count for every one of the six entities.
- [ ] Every Candidate row traces to a Bronze record hash (`_bronze_record_hash`).
- [ ] Safe-cast failures remain visible in the parse-failure checks, not silently dropped.
- [ ] The controlled repeat-run test (Part 10) shows stable counts.
- [ ] All six Candidate tables are Delta (Part 9.3).
- [ ] Student A, B and C can each explain the full flow and their owned deep area.
- [ ] No Week 6 or later implementation (DQ rules, quarantine, Trusted Silver, Gold, Power BI, streaming) has been added to this notebook.

**Quick viva**

1. Why does standardising a Candidate identifier not violate Bronze immutability?
2. Why is `TRY_CAST` used instead of a forcing `CAST` for every type conversion in this notebook?
3. What two checks together prove that records and lineage remain intact?
4. How are `calculated_net_payable` and `reserve_variance` different, and what does a non-zero `reserve_variance` mean?
5. Why are parse failures retained instead of corrected in Week 5?
6. Why do policies, policyholders, products, providers and claim_payments receive no invented calculated fields?

**System state after Week 5:** six Silver Candidate Delta tables
(`silver_claimiq_claims_candidate`, `silver_claimiq_policies_candidate`,
`silver_claimiq_policyholders_candidate`, `silver_claimiq_products_candidate`,
`silver_claimiq_providers_candidate`,
`silver_claimiq_claim_payments_candidate`), each typed, standardised,
row-count reconciled against Bronze and traceable by record hash. Week 6
applies documented data-quality rules to these Candidate tables and decides
which records become Trusted Silver and which require quarantine.